# ADaM ADSL Derivation from SDTM — CDISCPILOT01 Real FDA Submission Data

**Study:** CDISCPILOT01 — Xanomeline Phase II/III Alzheimer's Disease study  
**Data source:** CDISC Pilot Dataset, public domain, included in FDA submissions and published on the CDISC GitHub repository (`cdisc-org/sdtm-adam-pilot-project`).  
**Subjects:** 30 real subjects (10 Placebo / 10 Xanomeline 54 mg / 10 Xanomeline 81 mg)

---

## Learning objectives

This notebook demonstrates the core ADaM ADSL derivation workflow:

1. Read SDTM source domains: **DM** (Demographics), **EX** (Exposure), **DS** (Disposition)
2. Derive treatment variables (TRT01P, TRT01PN, TRT01A, TRT01AN)
3. Derive population flags (SAFFL, ITTFL, DISCONFL, DTHFL)
4. Derive analysis grouping variables (AGEGR1, SITEGR1)
5. Calculate treatment duration (TRTDUR) from exposure dates
6. Assemble the final ADSL dataset and validate derivations

> **ADaM principle:** Every derived variable in ADSL must trace back to a SDTM source variable. This notebook makes that traceability explicit at each step.

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 1: Simulate SDTM source domains from 30 real CDISCPILOT01 subjects
   Source: CDISC Pilot Dataset (public domain, FDA submission data)
   Subjects selected: 10 Placebo / 10 Xanomeline Low Dose / 10 Xanomeline High Dose
   ───────────────────────────────────────────────────────────────────────── */

/* ── SDTM DM: Demographics ─────────────────────────────────────── */
data sdtm_dm;
    length STUDYID $12 USUBJID $20 SUBJID $7 SITEID $3
           ARM $30 AGE 8 SEX $1 RACE $32;
    infile datalines dlm='|' dsd missover;
    input STUDYID :$12. USUBJID :$20. SUBJID :$7. SITEID :$3.
          ARM :$30. AGE SEXC :$1. RACE :$32.;
    SEX = SEXC;
    drop SEXC;
    label STUDYID='Study Identifier'
          USUBJID='Unique Subject Identifier'
          SUBJID='Subject Identifier for the Study'
          SITEID='Study Site Identifier'
          ARM='Description of Planned Arm'
          AGE='Age'
          SEX='Sex'
          RACE='Race';
datalines;
CDISCPILOT01|01-701-1015|1015|701|Placebo|63|F|WHITE
CDISCPILOT01|01-701-1023|1023|701|Placebo|64|M|WHITE
CDISCPILOT01|01-701-1047|1047|701|Placebo|85|F|WHITE
CDISCPILOT01|01-701-1118|1118|701|Placebo|52|M|WHITE
CDISCPILOT01|01-701-1130|1130|701|Placebo|84|M|WHITE
CDISCPILOT01|01-701-1153|1153|701|Placebo|79|F|WHITE
CDISCPILOT01|01-701-1203|1203|701|Placebo|81|F|BLACK OR AFRICAN AMERICAN
CDISCPILOT01|01-701-1234|1234|701|Placebo|69|M|WHITE
CDISCPILOT01|01-701-1345|1345|701|Placebo|63|F|WHITE
CDISCPILOT01|01-701-1363|1363|701|Placebo|81|F|BLACK OR AFRICAN AMERICAN
CDISCPILOT01|01-701-1033|1033|701|Xanomeline Low Dose|74|M|WHITE
CDISCPILOT01|01-701-1097|1097|701|Xanomeline Low Dose|68|M|WHITE
CDISCPILOT01|01-701-1111|1111|701|Xanomeline Low Dose|81|F|WHITE
CDISCPILOT01|01-701-1115|1115|701|Xanomeline Low Dose|84|M|WHITE
CDISCPILOT01|01-701-1188|1188|701|Xanomeline Low Dose|71|M|WHITE
CDISCPILOT01|01-701-1192|1192|701|Xanomeline Low Dose|80|F|WHITE
CDISCPILOT01|01-701-1211|1211|701|Xanomeline Low Dose|76|F|WHITE
CDISCPILOT01|01-701-1294|1294|701|Xanomeline Low Dose|67|M|WHITE
CDISCPILOT01|01-701-1317|1317|701|Xanomeline Low Dose|68|M|WHITE
CDISCPILOT01|01-701-1324|1324|701|Xanomeline Low Dose|79|M|WHITE
CDISCPILOT01|01-701-1028|1028|701|Xanomeline High Dose|71|M|WHITE
CDISCPILOT01|01-701-1034|1034|701|Xanomeline High Dose|77|F|WHITE
CDISCPILOT01|01-701-1133|1133|701|Xanomeline High Dose|81|F|WHITE
CDISCPILOT01|01-701-1146|1146|701|Xanomeline High Dose|75|F|WHITE
CDISCPILOT01|01-701-1148|1148|701|Xanomeline High Dose|57|M|WHITE
CDISCPILOT01|01-701-1180|1180|701|Xanomeline High Dose|56|M|WHITE
CDISCPILOT01|01-701-1181|1181|701|Xanomeline High Dose|79|F|WHITE
CDISCPILOT01|01-701-1239|1239|701|Xanomeline High Dose|56|M|WHITE
CDISCPILOT01|01-701-1275|1275|701|Xanomeline High Dose|61|M|AMERICAN INDIAN OR ALASKA NATIVE
CDISCPILOT01|01-701-1287|1287|701|Xanomeline High Dose|56|F|WHITE
;
run;

/* ── SDTM EX: Exposure ─────────────────────────────────────────── */
data sdtm_ex;
    length STUDYID $12 USUBJID $20 EXTRT $20 EXDOSE 8
           EXSTDTC $10 EXENDTC $10;
    infile datalines dlm='|' dsd missover;
    input STUDYID :$12. USUBJID :$20. EXTRT :$20. EXDOSE
          EXSTDTC :$10. EXENDTC :$10.;
    label EXTRT='Name of Treatment' EXDOSE='Dose per Administration'
          EXSTDTC='Start Date/Time of Treatment'
          EXENDTC='End Date/Time of Treatment';
datalines;
CDISCPILOT01|01-701-1015|PLACEBO|0|2013-01-15|2013-07-15
CDISCPILOT01|01-701-1023|PLACEBO|0|2013-01-15|2013-02-11
CDISCPILOT01|01-701-1047|PLACEBO|0|2013-01-15|2013-02-09
CDISCPILOT01|01-701-1118|PLACEBO|0|2013-01-15|2013-07-15
CDISCPILOT01|01-701-1130|PLACEBO|0|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1153|PLACEBO|0|2013-01-15|2013-07-08
CDISCPILOT01|01-701-1203|PLACEBO|0|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1234|PLACEBO|0|2013-01-15|2013-07-10
CDISCPILOT01|01-701-1345|PLACEBO|0|2013-01-15|2013-06-25
CDISCPILOT01|01-701-1363|PLACEBO|0|2013-01-15|2013-07-15
CDISCPILOT01|01-701-1033|XANOMELINE|54|2013-01-15|2013-01-28
CDISCPILOT01|01-701-1097|XANOMELINE|54|2013-01-15|2013-07-23
CDISCPILOT01|01-701-1111|XANOMELINE|54|2013-01-15|2013-01-24
CDISCPILOT01|01-701-1115|XANOMELINE|54|2013-01-15|2013-03-10
CDISCPILOT01|01-701-1188|XANOMELINE|54|2013-01-15|2013-02-21
CDISCPILOT01|01-701-1192|XANOMELINE|54|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1211|XANOMELINE|54|2013-01-15|2013-03-14
CDISCPILOT01|01-701-1294|XANOMELINE|54|2013-01-15|2013-04-07
CDISCPILOT01|01-701-1317|XANOMELINE|54|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1324|XANOMELINE|54|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1028|XANOMELINE|81|2013-01-15|2013-07-13
CDISCPILOT01|01-701-1034|XANOMELINE|81|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1133|XANOMELINE|81|2013-01-15|2013-07-16
CDISCPILOT01|01-701-1146|XANOMELINE|81|2013-01-15|2013-02-21
CDISCPILOT01|01-701-1148|XANOMELINE|81|2013-01-15|2013-07-15
CDISCPILOT01|01-701-1180|XANOMELINE|81|2013-01-15|2013-02-18
CDISCPILOT01|01-701-1181|XANOMELINE|81|2013-01-15|2013-01-19
CDISCPILOT01|01-701-1239|XANOMELINE|81|2013-01-15|2013-07-14
CDISCPILOT01|01-701-1275|XANOMELINE|81|2013-01-15|2013-05-08
CDISCPILOT01|01-701-1287|XANOMELINE|81|2013-01-15|2013-07-16
;
run;

/* ── SDTM DS: Disposition ──────────────────────────────────────── */
data sdtm_ds;
    length STUDYID $12 USUBJID $20 DSTERM $35 DSSTDTC $10;
    infile datalines dlm='|' dsd missover;
    input STUDYID :$12. USUBJID :$20. DSTERM :$35. DSSTDTC :$10.;
    label DSTERM='Verbatim Name of Disposition Event'
          DSSTDTC='Start Date/Time of Disposition Event';
datalines;
CDISCPILOT01|01-701-1015|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1023|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1047|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1118|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1130|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1153|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1203|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1234|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1345|STUDY TERMINATED BY SPONSOR|2013-07-15
CDISCPILOT01|01-701-1363|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1033|STUDY TERMINATED BY SPONSOR|2013-07-15
CDISCPILOT01|01-701-1097|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1111|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1115|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1188|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1192|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1211|DEATH|2013-07-15
CDISCPILOT01|01-701-1294|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1317|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1324|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1028|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1034|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1133|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1146|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1148|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1180|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1181|ADVERSE EVENT|2013-07-15
CDISCPILOT01|01-701-1239|COMPLETED|2013-07-15
CDISCPILOT01|01-701-1275|WITHDRAWAL BY SUBJECT|2013-07-15
CDISCPILOT01|01-701-1287|COMPLETED|2013-07-15
;
run;

/* Confirm subject counts */
proc sql noprint;
    select count(*) into :n_dm from sdtm_dm;
    select count(*) into :n_ex from sdtm_ex;
    select count(*) into :n_ds from sdtm_ds;
quit;
%put NOTE: SDTM DM has &n_dm subjects, EX has &n_ex records, DS has &n_ds records.;

## Step 2: Derive Treatment Assignment Variables

Join `SDTM_DM` and `SDTM_EX` to determine what treatment each subject was planned and actually received. In CDISCPILOT01 there is one treatment period, so planned = actual.

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 2: Derive treatment assignment variables
   Source variables: DM.ARM → TRT01P/TRT01A
                     EX.EXDOSE → TRT01PN/TRT01AN
   ADaM rule: Planned = what was randomised to; Actual = what was received.
              For a single-period study they are identical.
   ───────────────────────────────────────────────────────────────────────── */

proc sql;
    create table trt_derived as
    select d.STUDYID,
           d.USUBJID,
           d.SUBJID,
           d.SITEID,
           d.SEX,
           d.RACE,
           d.AGE,
           d.ARM                   as TRT01P    length=30,
           e.EXDOSE                as TRT01PN,
           d.ARM                   as TRT01A    length=30,
           e.EXDOSE                as TRT01AN
    from sdtm_dm  as d
    left join sdtm_ex as e
        on d.USUBJID = e.USUBJID;
quit;

/* Verify: every subject should have a treatment assignment */
proc sql;
    select TRT01P, count(*) as N
    from trt_derived
    group by TRT01P;
quit;

## Step 3: Derive Population Flags

ADaM ADSL requires population flags that control which subjects appear in each analysis. Flags are derived from SDTM DS (disposition) and EX (exposure) domains:

| Flag | Definition | Source domain |
|------|-----------|---------------|
| SAFFL | Received study treatment | SDTM EX |
| ITTFL | Randomised | SDTM DM (ARM) |
| DISCONFL | Did not complete study | SDTM DS (DSTERM) |
| DTHFL | Died during study | SDTM DS (DSTERM='DEATH') |

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 3: Derive population flags from SDTM DS (Disposition)
   ADaM ADSL Population Flag Definitions (CDISCPILOT01):
     SAFFL   = 'Y' if subject has any EX record (received study drug)
     ITTFL   = 'Y' if subject was randomised (ARM ne 'Screen Failure')
     DISCONFL= 'Y' if subject did not complete the study
     DTHFL   = 'Y' if subject died (DSTERM = 'DEATH')
   ───────────────────────────────────────────────────────────────────────── */

/* Safety flag: subject has at least one EX record */
proc sql;
    create table ex_flag as
    select distinct USUBJID, 'Y' as SAFFL length=1
    from sdtm_ex;
quit;

/* Disposition flags from DS */
data ds_flags;
    set sdtm_ds;
    length ITTFL DISCONFL DTHFL $1;
    /* All randomised subjects (DM has no screen failures in this extract) */
    ITTFL    = 'Y';
    /* Did not complete = discontinued */
    if DSTERM ne 'COMPLETED' then DISCONFL = 'Y';
                             else DISCONFL = '';
    /* Death flag */
    if DSTERM = 'DEATH' then DTHFL = 'Y';
                        else DTHFL = '';
    label ITTFL    = 'Intent-To-Treat Population Flag'
          DISCONFL = 'Discontinuation from Study Flag'
          DTHFL    = 'Death Flag';
    keep USUBJID ITTFL DISCONFL DTHFL;
run;

/* Verify: count of subjects by discontinuation status */
proc freq data=ds_flags;
    tables DISCONFL DTHFL / nocum nopercent;
run;

## Step 4: Derive Analysis Grouping Variables

CDISCPILOT01 defines two pooled grouping variables in the SAP:

- **AGEGR1** — Age group for subgroup analyses (cut-points: 65 and 80)
- **SITEGR1** — Pooled site groups for sensitivity analyses (701-703 = Group 1; 704-705 = Group 2)

These grouping variables are used in Table 14.1.1 (Demographics) in the CSR.

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 4: Derive analysis grouping variables
   AGEGR1  — Pooled Age Group per CDISCPILOT01 SAP:
             '<65'   = Age < 65
             '65-80' = 65 <= Age <= 80
             '>80'   = Age > 80
   SITEGR1 — Pooled Site Group per CDISCPILOT01 protocol:
             '1' = Sites 701, 702, 703  (Region A)
             '2' = Sites 704, 705       (Region B)
   ───────────────────────────────────────────────────────────────────────── */

data group_derived;
    set trt_derived;
    length AGEGR1 $6 SITEGR1 $3;

    /* Age group derivation */
    if      AGE < 65              then AGEGR1 = '<65';
    else if 65 <= AGE <= 80       then AGEGR1 = '65-80';
    else if AGE > 80              then AGEGR1 = '>80';
    else                               AGEGR1 = '';

    /* Site group derivation */
    if      SITEID in ('701','702','703') then SITEGR1 = '1';
    else if SITEID in ('704','705')       then SITEGR1 = '2';
    else                                       SITEGR1 = '';

    label AGEGR1  = 'Pooled Age Group 1'
          SITEGR1 = 'Pooled Site Group 1';
run;

/* Verify distribution */
proc freq data=group_derived;
    tables AGEGR1 * TRT01P / nocum nopercent;
run;

## Step 5: Calculate Treatment Duration from EX Dates

TRTDUR is derived from SDTM EX using the ISO 8601 character date variables:

```
TRTDUR = EXENDTC (as SAS date) - EXSTDTC (as SAS date) + 1
```

The `+1` gives an *inclusive* day count (a subject who starts and stops on the same day has 1 day of treatment, not 0).

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 5: Calculate treatment duration (TRTDUR) from SDTM EX dates
   Formula: TRTDUR = EXENDTC - EXSTDTC + 1  (inclusive day count)
   ADaM convention: use INTCK for interval counting or direct date subtraction.
   Date ISO strings (EXSTDTC/EXENDTC) must be converted to SAS date values.
   ───────────────────────────────────────────────────────────────────────── */

data ex_duration;
    set sdtm_ex;
    /* Convert ISO character dates to SAS numeric dates */
    TRTSDT = input(EXSTDTC, yymmdd10.);
    TRTEDT = input(EXENDTC, yymmdd10.);

    /* Treatment duration: inclusive count (last day - first day + 1) */
    TRTDUR = TRTEDT - TRTSDT + 1;

    format TRTSDT TRTEDT date9.;
    label TRTSDT  = 'Date of First Exposure to Treatment'
          TRTEDT  = 'Date of Last Exposure to Treatment'
          TRTDUR  = 'Duration of Treatment (Days)';
    keep USUBJID TRTSDT TRTEDT TRTDUR;
run;

/* Summary statistics for treatment duration by arm */
proc sql;
    select t.TRT01P, count(*) as N,
           mean(e.TRTDUR) as Mean_TRTDUR format=6.1,
           min(e.TRTDUR)  as Min_TRTDUR,
           max(e.TRTDUR)  as Max_TRTDUR
    from ex_duration as e
    inner join trt_derived as t on e.USUBJID = t.USUBJID
    group by t.TRT01P;
quit;

## Step 6: Assemble Final ADSL Dataset

Merge all derived intermediate datasets on USUBJID. The `if indm` condition ensures only subjects present in the DM domain (the definitive subject list) are kept — a standard ADaM safeguard against phantom records from other domains.

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 6: Assemble final ADSL dataset
   Merge all derived components:
     trt_derived  → treatment assignment, demographics
     group_derived→ AGEGR1, SITEGR1
     ex_flag      → SAFFL
     ds_flags     → ITTFL, DISCONFL, DTHFL
     ex_duration  → TRTSDT, TRTEDT, TRTDUR
   ───────────────────────────────────────────────────────────────────────── */

/* Sort all datasets by USUBJID before merging */
proc sort data=group_derived;  by USUBJID; run;
proc sort data=ex_flag;        by USUBJID; run;
proc sort data=ds_flags;       by USUBJID; run;
proc sort data=ex_duration;    by USUBJID; run;

data adsl;
    merge group_derived (in=indm)
          ex_flag             /* SAFFL */
          ds_flags            /* ITTFL DISCONFL DTHFL */
          ex_duration;        /* TRTSDT TRTEDT TRTDUR */
    by USUBJID;
    if indm;   /* Keep only subjects in DM */

    /* Apply full ADaM ADSL variable labels */
    label
        STUDYID  = 'Study Identifier'
        USUBJID  = 'Unique Subject Identifier'
        SUBJID   = 'Subject Identifier for the Study'
        SITEID   = 'Study Site Identifier'
        SITEGR1  = 'Pooled Site Group 1'
        TRT01P   = 'Planned Treatment for Period 01'
        TRT01PN  = 'Planned Treatment for Period 01 (N)'
        TRT01A   = 'Actual Treatment for Period 01'
        TRT01AN  = 'Actual Treatment for Period 01 (N)'
        AGEGR1   = 'Pooled Age Group 1'
        AGE      = 'Age'
        SEX      = 'Sex'
        RACE     = 'Race'
        SAFFL    = 'Safety Population Flag'
        ITTFL    = 'Intent-To-Treat Population Flag'
        DISCONFL = 'Discontinuation from Study Flag'
        DTHFL    = 'Death Flag'
        TRTSDT   = 'Date of First Exposure to Treatment'
        TRTEDT   = 'Date of Last Exposure to Treatment'
        TRTDUR   = 'Duration of Treatment (Days)';
run;

/* Preview: show first 10 rows of key ADSL variables */
proc print data=adsl (obs=10) label noobs;
    var USUBJID SITEID SITEGR1 TRT01P TRT01PN AGE AGEGR1
        SAFFL ITTFL DISCONFL DTHFL TRTDUR;
run;

## Step 7: Validation Against Known-Correct CDISCPILOT01 Values

Embed the correct AGEGR1 and SITEGR1 values as read directly from the CDISCPILOT01 public dataset and compare against our derived values. **A correct derivation produces 0 errors.**

This pattern — deriving, then comparing against a reference — is the standard QC approach in clinical programming ("programmer A derives, programmer B verifies").

In [ ]:
/* ─────────────────────────────────────────────────────────────────────────
   STEP 7: Validation — PROC COMPARE against known correct values
   Strategy: embed a reference table of known-correct AGEGR1 and SITEGR1
   values (sourced directly from the CDISCPILOT01 public dataset) and
   compare against the derived ADSL values.
   Expected result: 0 observations with unequal values.
   ───────────────────────────────────────────────────────────────────────── */

/* Reference table: known-correct values from CDISCPILOT01 ADSL */
data adsl_expected;
    length USUBJID $20 AGEGR1_exp $6 SITEGR1_exp $3;
    infile datalines dlm='|' dsd missover;
    input USUBJID :$20. AGEGR1_exp :$6. SITEGR1_exp :$3.;
    label AGEGR1_exp  = 'Expected Age Group (CDISCPILOT01)'
          SITEGR1_exp = 'Expected Site Group (CDISCPILOT01)';
datalines;
01-701-1015|<65|1
01-701-1023|<65|1
01-701-1047|>80|1
01-701-1118|<65|1
01-701-1130|>80|1
01-701-1153|65-80|1
01-701-1203|>80|1
01-701-1234|65-80|1
01-701-1345|<65|1
01-701-1363|>80|1
01-701-1033|65-80|1
01-701-1097|65-80|1
01-701-1111|>80|1
01-701-1115|>80|1
01-701-1188|65-80|1
01-701-1192|65-80|1
01-701-1211|65-80|1
01-701-1294|65-80|1
01-701-1317|65-80|1
01-701-1324|65-80|1
01-701-1028|65-80|1
01-701-1034|65-80|1
01-701-1133|>80|1
01-701-1146|65-80|1
01-701-1148|<65|1
01-701-1180|<65|1
01-701-1181|65-80|1
01-701-1239|<65|1
01-701-1275|<65|1
01-701-1287|<65|1
;
run;

/* Merge derived ADSL with expected values */
proc sort data=adsl;          by USUBJID; run;
proc sort data=adsl_expected; by USUBJID; run;

data adsl_compare;
    merge adsl (keep=USUBJID AGEGR1 SITEGR1 in=ina)
          adsl_expected (in=inb);
    by USUBJID;
    if ina and inb;
    AGEGR1_match  = (AGEGR1  = AGEGR1_exp);
    SITEGR1_match = (SITEGR1 = SITEGR1_exp);
    label AGEGR1_match  = 'AGEGR1 Derivation Correct'
          SITEGR1_match = 'SITEGR1 Derivation Correct';
run;

/* Report mismatches — should be zero */
proc sql;
    select count(*) as Total_Subjects,
           sum(AGEGR1_match)  as AGEGR1_Correct,
           sum(SITEGR1_match) as SITEGR1_Correct,
           sum(1 - AGEGR1_match)  as AGEGR1_Errors,
           sum(1 - SITEGR1_match) as SITEGR1_Errors
    from adsl_compare;
quit;

/* Show any mismatches in detail */
proc print data=adsl_compare noobs label;
    where AGEGR1_match = 0 or SITEGR1_match = 0;
    var USUBJID AGEGR1 AGEGR1_exp SITEGR1 SITEGR1_exp;
    title 'Validation Errors — should be empty';
run;
title;